# Lab 01a: A Ilusão da Velocidade (Speedup)

**Objetivo:** Medir o ganho real de performance ao paralelizar uma tarefa de processamento intensivo de CPU/GPU e entender a necessidade vital de clusters para cálculos de risco bancário.
**Ambiente:** Google Colab.
**Cenário:** Simulação de Monte Carlo para Cálculo de Risco em 500 Propostas de Crédito de Alto Valor (CAIXA).

## 🚀 Passo 0: Preparação do Ambiente (GPU)

Para este laboratório, precisamos garantir que o Google Colab nos forneceu uma GPU.

1. No menu superior, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução**.
2. Em "Acelerador de hardware", selecione **T4 GPU**.
3. Clique em **Salvar**.
4. Execute a célula abaixo para validar.

In [ ]:
!nvidia-smi

## 🛠️ Passo 1: O Cenário (Execução Serial - CPU)

Vamos simular o **Risco de Crédito**. Para cada proposta de empréstimo, realizaremos **20.000.000 (20 milhões)** de simulações de cenários econômicos. Esta é uma tarefa puramente **CPU Bound**.

In [ ]:
import time
import numpy as np
import torch
import os

# Simulação de 500 propostas de alto valor
propostas = [{"id": i, "valor": np.random.randint(100000, 1000000)} for i in range(500)]

def simular_risco_monte_carlo(proposta):
    # 20.000.000 de iterações garante ~6 a 8 minutos de execução serial no Colab
    n_cenarios = 20000000 
    
    # Geração de cenários de inadimplência
    cenarios = np.random.normal(0.05, 0.15, n_cenarios)
    
    # Cálculo de Perda com Desconto de Valor Presente (NPV)
    perdas_brutas = proposta['valor'] * (cenarios[cenarios > 0.20])
    taxas_desconto = np.random.uniform(0.08, 0.12, len(perdas_brutas))
    perdas_descontadas = perdas_brutas / (1 + taxas_desconto)
    
    return np.mean(perdas_descontadas) if len(perdas_descontadas) > 0 else 0

# Execução Serial (Baseline)
print(f"Iniciando simulação serial para {len(propostas)} propostas... Isso levará aprox. 6-7 minutos.")
start_time = time.time()

resultados_seriais = [simular_risco_monte_carlo(p) for p in propostas]

tempo_serial = time.time() - start_time
print(f"--- Execução Serial Finalizada ---")
print(f"Tempo Total: {tempo_serial:.2f} segundos (~{tempo_serial/60:.1f} min)")

### 🧠 Provocação 01 (Cenário Serial):

**Pergunta:** Se a CAIXA recebe **500.000 contratos** para análise em um único dia (ex: mutirão de crédito), quanto tempo levaria para processar tudo utilizando apenas 1 Core de CPU com este script?

*Responda abaixo fazendo o cálculo matemático baseado no resultado que você obteve.*

## 🚀 Passo 2: Aceleração Paralela (Multi-CPU)

Utilizaremos o `ProcessPoolExecutor` para distribuir as propostas entre os núcleos da CPU do Google Colab.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

n_cores = os.cpu_count()
print(f"Núcleos disponíveis: {n_cores}")

print(f"Iniciando simulação paralela em {n_cores} núcleos...")
start_time = time.time()

with ProcessPoolExecutor(max_workers=n_cores) as executor:
    resultados_paralelos = list(executor.map(simular_risco_monte_carlo, propostas))

tempo_paralelo = time.time() - start_time
print(f"--- Execução Paralela (CPU) Finalizada ---")
print(f"Tempo Total: {tempo_paralelo:.2f} segundos")

### 🧠 Provocação 02 (Cenário Paralelo):

**Pergunta:** O seu ganho foi de exatamente 2x (se o Colab te deu 2 cores)? Se não, onde o tempo foi "perdido"? Pense no que o Python precisa fazer para que um processo "fale" com o outro (IPC).

**Desafio:** Se você tivesse 64 núcleos, o tempo cairia para 1/64 do original? Por que a **Lei de Amdahl** diz que isso é impossível?

## ⚡ Passo 3: O Salto Tecnológico (Aceleração por GPU)

Diferente da CPU, a **GPU T4** possui **2.560 núcleos**. Vamos usar o **PyTorch** para processar a matriz massiva de cenários.

In [ ]:
def simular_gpu(propostas_lista):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Iniciando simulação em GPU ({device}) via PyTorch...")
    
    # Extraindo valores para um Tensor (Vetor da GPU)
    valores = torch.tensor([p['valor'] for p in propostas_lista], device=device).view(-1, 1)
    
    # Para a GPU ser eficiente, processamos 1 milhão de cenários por contrato de uma só vez
    n_cenarios_gpu = 1000000 
    n_propostas = len(propostas_lista)
    
    start_time_gpu = time.time()
    
    # Gerando cenários para TODAS as 500 propostas de uma vez (Matriz de 500 Milhões de elementos)
    cenarios = torch.randn((n_propostas, n_cenarios_gpu), device=device) * 0.15 + 0.05
    
    # Cálculo Vetorial Massivo na GPU
    mask = cenarios > 0.20
    perdas_brutas = valores * mask.float()
    resultados_gpu = perdas_brutas.mean(dim=1)
    
    # Sincroniza a GPU para medição real de tempo
    torch.cuda.synchronize()
    
    tempo_final_gpu = time.time() - start_time_gpu
    return resultados_gpu, tempo_final_gpu

res_gpu, tempo_gpu = simular_gpu(propostas)
print(f"--- Execução GPU Finalizada ---")
print(f"Tempo Total: {tempo_gpu:.2f} segundos")

### 🧠 Provocação 03 (Cenário GPU):

**Pergunta:** A GPU foi centenas de vezes mais rápida. Se ela é tão superior, por que ainda usamos CPUs para processamento distribuído (como no Spark)?

**Dica:** O que acontece se o dado de entrada (propostas) for um arquivo Parquet de **1 Terabyte** que não cabe na memória (VRAM) de 16GB da GPU?

## 📊 Passo 4: Comparativo Final

Execute a célula abaixo para ver o ranking de performance.

In [ ]:
print(f"--- QUADRO COMPARATIVO ---")
print(f"1. Serial (CPU):   {tempo_serial:.2f}s")
print(f"2. Paralelo (CPU): {tempo_paralelo:.2f}s (Speedup: {tempo_serial/tempo_paralelo:.2f}x)")
print(f"3. GPU (PyTorch):  {tempo_gpu:.2f}s (Speedup: {tempo_serial/tempo_gpu:.2f}x)")
print("\n--- CONCLUSÃO DE ENGENHARIA ---")
if tempo_gpu < tempo_paralelo:
    print("Vencemos o tempo! Mas note: para 500 contratos a GPU brilhou. Para 500 milhões de contratos, precisaremos de CLUSTERS (Próxima Aula).")